In [1]:
from shapely.geometry import box, Point, LineString, Polygon, MultiPolygon
from shapely.ops import unary_union
import geopandas as gpd
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import rasterio
from rasterio.mask import mask
from rasterio.plot import show
from rasterio.warp import calculate_default_transform, reproject
from rasterio.enums import Resampling
import osmnx as ox  
import contextily as ctx
from geopy.geocoders import Nominatim  
from geopy.extra.rate_limiter import RateLimiter
import os
#!pip install h3
import h3
import matplotlib.patches as mpatches

In [2]:
# Get the current working directory
current_path = os.getcwd()

# Define relative paths for data, output, and helper functions
data_in = os.path.join(current_path, "data")       # Folder with input data
output = os.path.join(current_path, "output")      # Folder for saving results

In [9]:
data_ssi = gpd.read_file('data/data_ssi.geojson')
print(data_ssi.crs)
data_ssi = data_ssi.to_crs(epsg=31256)
data_ssi['area_ssi'] = data_ssi.geometry.area
data_ssi = data_ssi[['ZBEZ', 'area_ssi']]
data_ssi.head(5)

EPSG:4326


,ZBEZ,area_ssi
0,0101,377557.211380
1,0102,379376.751386
2,0103,465496.860383
3,0104,835823.045243
4,0105,333293.038103


In [10]:
boundaries_path = 'data/ZAEHLBEZIRKOGDPolygon.shp'
census_boundaries = gpd.read_file(boundaries_path)
census_boundaries = census_boundaries.to_crs(epsg=31256)
census_boundaries['area_boundaries'] = census_boundaries.geometry.area
census_boundaries = census_boundaries[['ZBEZ', 'FLAECHE', 'area_boundaries']]
census_boundaries.head(3)

,ZBEZ,FLAECHE,area_boundaries
0,2219,431562.4878,431562.489038
1,1909,700939.3051,700939.307083
2,0902,471286.9574,471286.958700


In [17]:
df_full = pd.merge(data_ssi, census_boundaries, on='ZBEZ', how='outer')
df_full.head(100)

,ZBEZ,area_ssi,FLAECHE,area_boundaries
0,0101,3.775572e+05,3.774617e+05,3.774617e+05
1,0102,3.793768e+05,3.793768e+05,3.793768e+05
2,0103,4.654969e+05,4.654969e+05,4.654969e+05
3,0104,8.358230e+05,8.358230e+05,8.358230e+05
4,0105,3.332930e+05,3.332930e+05,3.332930e+05
...,...,...,...,...
95,1207,8.403577e+05,8.406796e+05,8.406796e+05
96,1208,6.670974e+05,6.664961e+05,6.664961e+05
97,1209,5.795694e+05,5.795694e+05,5.795694e+05
98,1210,1.401321e+06,1.401321e+06,1.401321e+06


In [13]:
df_full[['area_ssi', 'FLAECHE', 'area_boundaries']].sum()

area_ssi           3.633159e+08
FLAECHE            4.148706e+08
area_boundaries    4.148706e+08
dtype: float64

In [23]:
df_full['abs_diff'] = (df_full['area_boundaries'] - df_full['area_ssi']).abs()
df_full = df_full.sort_values('abs_diff', ascending=False)
df_full.head()
df_full['abs_diff'].sum()

np.float64(94174.43502566064)

In [26]:
print(df_full.isnull().sum())

ZBEZ               0
area_ssi           4
FLAECHE            0
area_boundaries    0
area_diff          4
abs_diff           4
dtype: int64


In [28]:
print(df_full[df_full['area_ssi'].isnull()])

     ZBEZ  area_ssi       FLAECHE  area_boundaries  area_diff  abs_diff
16   0210       NaN  1.250098e+06     1.250098e+06        NaN       NaN
109  1310       NaN  2.349476e+07     2.349476e+07        NaN       NaN
198  2130       NaN  7.276094e+05     7.276095e+05        NaN       NaN
228  2230       NaN  2.608180e+07     2.608180e+07        NaN       NaN


In [36]:
data_ssi2 = gpd.read_file('data/data_ssi.geojson')
print(data_ssi2.crs)
data_ssi2 = data_ssi2.to_crs(epsg=31256)
data_ssi2['area_ssi'] = data_ssi2.geometry.area
data_ssi2[data_ssi2['ZBEZ'].isin(['0101','0210', '1310', '2130', '2230'])]

EPSG:4326


,ZBEZ,university_share,avg_income,unemprate,benefit_share,SSI,quintile,geometry,area_ssi
0,0101,1.604954,1.741399,1.125923,1.123008,1.653992,5,"POLYGON ((3485.019 341498.126, 3484.459 341498...",377557.21138
